# Combine files

In [23]:
import os
import re
import glob
import pandas as pd
from collections import defaultdict

## Find lack company code compared to 2025 (Get new file) ----------------------------------------------------------------------

### Remove unecessary code 

In [73]:
def read_file(path):
    return pd.read_csv(
        path,
        encoding="cp932",
        sep=",",
        on_bad_lines="skip",
        dtype={"組織コード": str},
        keep_default_na=False
    )

ref_2025 = {
 '003000000': 'イノチオアグリ㈱',
 '003090100': 'イノチオアグリ本社',
 '230B30300': 'システム設計課',
 '230J30120': 'フィルム加工課 出荷部門',
 '230J30110': 'フィルム加工課 生産部門',
 '230J30100': 'フィルム加工課',
 '230H10200': 'マーケティング課',
 '230C30600': 'メンテナンス課',
 '230H62000': '人財育成課',
 '230H61600': '企画部',
 '230C60100': '品質管理課',
 '230H10100': '営業推進室',
 '230B10100': '営農支援部',
 '040A10100': '宮城営業チーム',
 '040A10300': '宮城営業課',
 '040A30101': '宮城工務チーム',
 '230S10200': '尾張営業チーム',
 '230S10300': '尾張営業課',
 '230S30101': '尾張工務チーム',
 '230C30500': '工務管理課',
 '230C30400': '工務統括部',
 '400A10100': '広域支援課',
 '003030100': '技術本部',
 '230C30310': '施設加工課',
 '003090200': '本社収支',
 '230H10500': '本社営業チーム',
 '230H10600': '本社営業課',
 '230H30200': '本社工務チーム',
 '230P10100': '東海支援課',
 '220C10100': '浜松営業チーム',
 '220C10200': '浜松営業課',
 '220C30101': '浜松工務チーム',
 '230A10200': '渥美営業チーム',
 '230A10300': '渥美営業課',
 '230A30101': '渥美工務チーム',
 '230B60100': '物流チーム',
 '230B10200': '田原営業チーム',
 '230B10300': '田原営業課',
 '230B30601': '田原工務チーム',
 '400A10200': '福岡営業チーム',
 '400A10300': '福岡営業課',
 '400A30100': '福岡工務チーム',
 '040A10200': '福島営業チーム',
 '040A10400': '福島営業課',
 '040A30201': '福島工務チーム',
 '230H61210': '管理課',
 '230H61100': '管理部',
 '003060100': '経営本部',
 '230C30200': '製造部',
 '230Q10200': '西三河営業チーム',
 '230Q10300': '西三河営業課',
 '230Q30101': '西三河工務チーム',
 '230B30200': '設計課',
 '230B30100': '設計開発部',
 '230H61500': '調達チーム',
 '230P10300': '豊川営業チーム',
 '230P10400': '豊川営業課',
 '230P30101': '豊川工務チーム',
 '230K10100': '豊橋営業チーム',
 '230K10200': '豊橋営業課',
 '230K30101': '豊橋工務チーム',
 '230C30330': '資材物流課',
 '230H61800': '購買課',
 '230K10300': '農薬推進課',
 '230B50100': '開発課',
 '100A10200': '関東営業チーム',
 '100A10300': '関東営業課',
 '100A30101': '関東工務チーム'
}
# Read 2025.txt
actual_2025_df = read_file(r"C:\Users\2372\Downloads\2025.txt")
actual_2025 = dict(zip(actual_2025_df["組織コード"], actual_2025_df["組織名"]))

ref_codes = set(ref_2025.keys())
actual_codes = set(actual_2025.keys())

only_in_ref = ref_codes - actual_codes
only_in_actual = actual_codes - ref_codes
common_codes = ref_codes & actual_codes

name_diff = {
    code: {"ref": ref_2025[code], "actual": actual_2025[code]}
    for code in common_codes
    if ref_2025[code] != actual_2025[code]
}

print(f"Only in reference (missing in file): {sorted(only_in_ref)}")
print(f"\nOnly in file (extra codes): {sorted(only_in_actual)}")
print(f"\nName mismatches ({len(name_diff)}):")
for code, diff in sorted(name_diff.items()):
    print(f"  {code}: ref='{diff['ref']}' | actual='{diff['actual']}'")

Only in reference (missing in file): []

Only in file (extra codes): ['003010100', '003090300', '003090400', '003090500', '003091300', '003091500', '003092100', '003095310', '003095320', '003095410', '003095420', '100A10100', '210A10200', '230A10100', '230H10400']

Name mismatches (0):


In [50]:
len(only_in_actual)

4

### Count rows that contain codes in only_in_actual for each file in file_paths

In [74]:
target_codes = {str(c).strip() for c in only_in_actual}

file_paths = [
    r"C:\Users\2372\Downloads\2022.txt",
    r"C:\Users\2372\Downloads\2023.txt",
    r"C:\Users\2372\Downloads\2024.txt",
    r"C:\Users\2372\Downloads\2025.txt",
    r"C:\Users\2372\Downloads\2026.txt",
]

count_rows = []
detail_rows = []

for path in file_paths:
    df_tmp = pd.read_csv(
        path,
        encoding="cp932",
        sep=",",
        on_bad_lines="skip",
        dtype={"組織コード": str},
        keep_default_na=False
    )
    df_tmp.columns = [c.strip() for c in df_tmp.columns]

    if "組織コード" not in df_tmp.columns:
        count_rows.append({"file": os.path.basename(path), "matched_rows": None, "note": "組織コード column not found"})
        continue

    df_tmp["組織コード"] = df_tmp["組織コード"].astype(str).str.strip()
    matched = df_tmp[df_tmp["組織コード"].isin(target_codes)]

    count_rows.append({
        "file": os.path.basename(path),
        "matched_rows": len(matched),
        "matched_unique_codes": matched["組織コード"].nunique()
    })

    if not matched.empty:
        per_code = matched["組織コード"].value_counts().rename_axis("組織コード").reset_index(name="row_count")
        per_code["file"] = os.path.basename(path)
        detail_rows.append(per_code)

count_df = pd.DataFrame(count_rows)
display(count_df)

if detail_rows:
    detail_df = pd.concat(detail_rows, ignore_index=True)[["file", "組織コード", "row_count"]]
    detail_df = detail_df.sort_values(["file", "組織コード"]).reset_index(drop=True)
    display(detail_df)

,file,matched_rows,matched_unique_codes
0,2022.txt,14,14
1,2023.txt,14,14
2,2024.txt,14,14
3,2025.txt,15,15
4,2026.txt,14,14


,file,組織コード,row_count
0,2022.txt,003010100,1
1,2022.txt,003090300,1
2,2022.txt,003090400,1
3,2022.txt,003090500,1
4,2022.txt,003091300,1
...,...,...,...
66,2026.txt,003095410,1
67,2026.txt,003095420,1
68,2026.txt,100A10100,1
69,2026.txt,210A10200,1


### Remove rows by 組織コード (using existing codes_to_remove and file_paths)

In [75]:
target_codes = {str(c).strip() for c in only_in_actual}

file_paths = [
    r"C:\Users\2372\Downloads\2022.txt",
    r"C:\Users\2372\Downloads\2023.txt",
    r"C:\Users\2372\Downloads\2024.txt",
    r"C:\Users\2372\Downloads\2025.txt",
    r"C:\Users\2372\Downloads\2026.txt",
]

for path in file_paths:
    df_tmp = pd.read_csv(
        path,
        encoding="cp932",
        sep=",",
        on_bad_lines="skip",
        dtype={"組織コード": str},
        keep_default_na=False
    )
    df_tmp.columns = [c.strip() for c in df_tmp.columns]

    if "組織コード" not in df_tmp.columns:
        print(f"{path}: skipped (no '組織コード' column)")
        continue

    before = len(df_tmp)
    df_tmp["組織コード"] = df_tmp["組織コード"].astype(str).str.strip()
    df_tmp = df_tmp[~df_tmp["組織コード"].isin(target_codes)].copy()
    removed = before - len(df_tmp)

    df_tmp.to_csv(path, index=False, encoding="cp932")
    print(f"{path}: removed {removed} rows, remaining {len(df_tmp)}")

C:\Users\2372\Downloads\2022.txt: removed 14 rows, remaining 71
C:\Users\2372\Downloads\2023.txt: removed 14 rows, remaining 73
C:\Users\2372\Downloads\2024.txt: removed 14 rows, remaining 68
C:\Users\2372\Downloads\2025.txt: removed 15 rows, remaining 68
C:\Users\2372\Downloads\2026.txt: removed 14 rows, remaining 71


### Remove row where 合計算入区分 = 0

In [76]:
for path in file_paths:
    df_tmp = pd.read_csv(
        path,
        encoding="cp932",
        sep=",",
        on_bad_lines="skip",
        keep_default_na=False
    )
    df_tmp.columns = [c.strip() for c in df_tmp.columns]

    if "合計算入区分" not in df_tmp.columns:
        print(f"{path}: skipped (no '合計算入区分' column)")
        continue

    before = len(df_tmp)
    kbn = pd.to_numeric(df_tmp["合計算入区分"], errors="coerce")
    df_tmp = df_tmp[kbn != 0].copy()
    removed = before - len(df_tmp)

    df_tmp.to_csv(path, index=False, encoding="cp932")
    print(f"{path}: removed {removed} rows where 合計算入区分=0, remaining {len(df_tmp)}")

C:\Users\2372\Downloads\2022.txt: removed 0 rows where 合計算入区分=0, remaining 71
C:\Users\2372\Downloads\2023.txt: removed 0 rows where 合計算入区分=0, remaining 73
C:\Users\2372\Downloads\2024.txt: removed 0 rows where 合計算入区分=0, remaining 68
C:\Users\2372\Downloads\2025.txt: removed 0 rows where 合計算入区分=0, remaining 68
C:\Users\2372\Downloads\2026.txt: removed 0 rows where 合計算入区分=0, remaining 71


In [50]:
df = pd.read_csv(
        file_paths[0],
        encoding="cp932",
        sep=",",
        on_bad_lines="skip",
        keep_default_na=False
    )
df

,対象年月度,組織コード,組織名,組織名（カナ）,上位組織コード,組織区分,事業所コード,管轄コード,合計算入区分,実績計上区分,出力順,組織ランク,内線電話,責任者名,帳票出力区分,開始日,停止日
0,202303,001000000,イノチオホールディングス㈱,,,5,230H,0,1,0,1,1,,石黒 功,1,2022/02/01,
1,202303,001050100,研究開発本部,,001000000,3,230G,0,1,0,13,2,,梅村 賢司,1,2022/02/01,
2,202303,001060100,管理本部,,001000000,4,230H,0,1,0,2,2,,林 耕一,1,2022/02/01,
3,202303,001090100,イノチオホールディングス本社,,001000000,4,230H,0,1,1,23,2,,,1,2022/02/03,
4,202303,001090200,本社収支,,001000000,4,230H,0,1,1,24,2,,,1,2022/02/03,
5,202303,001091400,中央農業研究所共通,,,4,230G,0,1,1,26,4,,,0,2022/02/03,
6,202303,001091500,高師事業所共通,,,4,230H,0,1,1,25,4,,,0,2022/02/03,
7,202303,001095100,不動産部門,,001000000,4,230H,0,1,1,19,2,,村井 悟,1,2022/02/01,
8,202303,001095200,システム部門,,001000000,4,230H,0,1,1,20,2,,倉員 武志,1,2022/02/02,
9,202303,001095500,事業継承ビジネス,,001000000,4,230H,0,1,1,21,2,,,1,2022/02/03,


In [53]:
df = df[["組織コード", "組織名"]]

In [ ]:
output_txt_path = os.path.join(r"C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績", "df_export.txt")
df.to_csv(output_txt_path, index=False, sep=",", encoding="cp932")
print(f"Saved: {output_txt_path}")

Saved: C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\df_export.txt


: 

### Check difference based on date of a file

In [77]:
import os
import re
import pandas as pd

# =========================
# Input / Output
# =========================
file_paths = [
    r"C:\Users\2372\Downloads\2022.txt",
    r"C:\Users\2372\Downloads\2023.txt",
    r"C:\Users\2372\Downloads\2024.txt",
    r"C:\Users\2372\Downloads\2025.txt",
    r"C:\Users\2372\Downloads\2026.txt",
]

output_dir = r"C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績"
output_path = os.path.join(output_dir, "diff_vs_2025_by_year_ONLY_DIFF_AGRI.txt")


# -------------------------
# Helpers
# -------------------------
def extract_year(path: str) -> str:
    m = re.search(r"(20\d{2})", os.path.basename(path))
    return m.group(1) if m else os.path.basename(path)

def read_year_file(path: str) -> pd.DataFrame:
    df = pd.read_csv(
        path,
        encoding="cp932",
        sep=",",
        on_bad_lines="skip",
        dtype={"組織コード": str},
        keep_default_na=False
    )
    df.columns = [c.strip() for c in df.columns]

    required = {"組織コード", "組織名"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns {missing}. Found: {list(df.columns)}")

    df = df[["組織コード", "組織名"]].copy()
    df["組織コード"] = df["組織コード"].astype(str).str.strip()
    df["組織名"] = df["組織名"].astype(str).str.strip()

    df = df[df["組織コード"] != ""]
    return df.drop_duplicates()

def build_maps(df: pd.DataFrame):
    blank_name_codes = set(df.loc[df["組織名"] == "", "組織コード"].unique().tolist())

    df_nonblank = df[df["組織名"] != ""]
    code_to_name = (
        df_nonblank.groupby("組織コード")["組織名"]
        .agg(lambda s: " / ".join(sorted(set(s))))
        .to_dict()
    )

    name_to_codes = {}
    for c, nm_joined in code_to_name.items():
        for nm in [x.strip() for x in nm_joined.split("/")]:
            if nm:
                name_to_codes.setdefault(nm, set()).add(c)

    return code_to_name, name_to_codes, blank_name_codes


# -------------------------
# Load all years
# -------------------------
year_data = {}
errors = []

for p in file_paths:
    y = extract_year(p)
    try:
        df = read_year_file(p)
        code_to_name, name_to_codes, blank_name_codes = build_maps(df)
        year_data[y] = {
            "code_to_name": code_to_name,
            "name_to_codes": name_to_codes,
            "blank_name_codes": blank_name_codes
        }
    except Exception as e:
        errors.append((p, str(e)))

years_sorted = sorted(year_data.keys())

if "2025" not in year_data:
    raise ValueError("2025 file is required as the reference year.")

ref_2025 = year_data["2025"]["code_to_name"]
ref_2025_names = set(ref_2025.values())
ref_2025_codes = set(ref_2025.keys())


# -------------------------
# Build output
# -------------------------
lines = []
lines.append("=== Compare vs 2025 (BIDIRECTIONAL) ===")
lines.append("A) Codes in 2025 that are changed or missing in another year")
lines.append("B) Codes in another year that do not exist in 2025")
lines.append("C) Names in another year that do not exist in 2025")
lines.append("")

if errors:
    lines.append("=== Read errors ===")
    for p, e in errors:
        lines.append(f"- {p}: {e}")
    lines.append("")

for y in years_sorted:
    if y == "2025":
        continue

    code_to_name = year_data[y]["code_to_name"]
    name_to_codes = year_data[y]["name_to_codes"]
    blank_name_codes = year_data[y]["blank_name_codes"]

    year_codes = set(code_to_name.keys()) | set(blank_name_codes)
    year_names = set(code_to_name.values())

    lines.append(f"==================== {y} vs 2025 ====================")

    # -----------------------------------
    # A) 2025 side: changed / missing
    # -----------------------------------
    lines.append("[A] 2025 codes changed or missing in this year")
    diff_dict = {}

    for code, base_name in ref_2025.items():
        yr_name = code_to_name.get(code, None)

        if yr_name is not None:
            if yr_name != base_name:
                diff_dict[code] = f"2025='{base_name}' | {y}='{yr_name}'"
            continue

        if code in blank_name_codes:
            candidates = sorted(name_to_codes.get(base_name, set()))
            if candidates:
                diff_dict[code] = (
                    f"2025='{base_name}' | {y}='__MISSING_NAME__' "
                    f"(same name found under codes: {', '.join(candidates[:20])}"
                    f"{' ...' if len(candidates) > 20 else ''})"
                )
            else:
                diff_dict[code] = f"2025='{base_name}' | {y}='__MISSING_NAME__'"
            continue

        candidates = sorted(name_to_codes.get(base_name, set()))
        if candidates:
            diff_dict[code] = (
                f"2025='{base_name}' | {y}='__MISSING__' "
                f"(same name found under codes: {', '.join(candidates[:20])}"
                f"{' ...' if len(candidates) > 20 else ''})"
            )
        else:
            diff_dict[code] = f"2025='{base_name}' | {y}='__MISSING__'"

    if diff_dict:
        for code in sorted(diff_dict.keys()):
            lines.append(f"  {code}: {diff_dict[code]}")
    else:
        lines.append("  None")
    lines.append("")

    # -----------------------------------
    # B) Other year side: extra codes
    # -----------------------------------
    lines.append(f"[B] Codes in {y} but not in 2025")
    extra_codes = sorted(year_codes - ref_2025_codes)

    if extra_codes:
        for code in extra_codes:
            if code in code_to_name:
                nm = code_to_name[code]
                lines.append(f"  {code}: {nm}")
            elif code in blank_name_codes:
                lines.append(f"  {code}: __BLANK_NAME__")
    else:
        lines.append("  None")
    lines.append("")

    # -----------------------------------
    # C) Other year side: extra names
    # -----------------------------------
    lines.append(f"[C] Names in {y} but not in 2025")
    extra_names = sorted(year_names - ref_2025_names)

    if extra_names:
        for nm in extra_names:
            codes = sorted(name_to_codes.get(nm, []))
            lines.append(f"  {nm}: codes={', '.join(codes)}")
    else:
        lines.append("  None")
    lines.append("")

# -------------------------
# Save
# -------------------------
os.makedirs(output_dir, exist_ok=True)
with open(output_path, "w", encoding="utf-8") as f:
    f.write("\n".join(lines))

print(f"Saved to: {output_path}")

Saved to: C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\diff_vs_2025_by_year_ONLY_DIFF_AGRI.txt


## -----------------------------END-------------------------------------

## Delete all OLD csv files in Lates Date folder

In [2]:
import os
import glob

# Path to the parent folder
parent_folder = r"C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\Latest Date"

# Find all CSV files in all subfolders
csv_files = glob.glob(f"{parent_folder}\\**\\*.csv", recursive=True)

# Delete each CSV file
for file_path in csv_files:
    try:
        os.remove(file_path)
        print(f"Deleted: {file_path}")
    except Exception as e:
        print(f"Failed to delete {file_path}: {e}")

print(f"\nTotal CSV files deleted: {len(csv_files)}")

Deleted: C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\Latest Date\みらい\PKTR022_みらい_202606.csv
Deleted: C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\Latest Date\アグリ\PKTR022_アグリ_202606.csv
Deleted: C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\Latest Date\アグリコ\PKTR022_アグリコ_202606.csv
Deleted: C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\Latest Date\イノチオ物流\PKTR022_物流_202606.csv
Deleted: C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\Latest Date\スズキヤングプランツ\PKTR022_ヤング_202606.csv
Deleted: C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\Latest Date\フジプランツ\PKTR022_フジプランツ_202606.csv
Deleted: C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\Latest Date\フローラ\PKTR022_フローラ_202606.csv
Deleted: C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\Latest Date\プラントケア\PKTR022_プラントケア_202606.csv
Deleted: C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\Latest Date\ランドサイエンス\PKTR022_ランドサイエンス_202606.csv
Deleted: C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_R

## Combine all files in a folder

In [54]:
base_folder = r'C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\Latest Date\農芸'

# # Tìm tất cả các file .txt trong thư mục gốc và các thư mục con
file_list = glob.glob(f"{base_folder}\\*.txt")

# Các cột cần ép kiểu float
float_columns = ['M/P', '予定', '実績', '構成比(%)', '予定比(%)', 'M/P比(%)']

# Đọc và nối tất cả các file
dfs = []
for f in file_list:
    with open(f, 'r', encoding='cp932', errors='ignore') as file:
        if '<!DOCTYPE HTML' in file.read():
            print(f"Bỏ qua file HTML: {f}")
            continue

    df = pd.read_csv(f, encoding="cp932", delimiter='\t', on_bad_lines='skip', dtype={'組織コード': str})
    for col in float_columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col].replace('-', 0), errors='coerce')
    dfs.append(df)

df = pd.concat(dfs, ignore_index=True)
df
# Hiển thị số lượng file đã đọc và số dòng tổng cộng
print(f"Đã đọc {len(file_list)} file.")
print(f"Tổng số dòng: {len(df)}")

# Delete all files in the base_folder
for filename in os.listdir(base_folder):
    file_path = os.path.join(base_folder, filename)
    if os.path.isfile(file_path):
        os.remove(file_path)

print("All files in the folder have been deleted.")

Đã đọc 5 file.
Tổng số dòng: 385
All files in the folder have been deleted.


In [55]:
file_latest = r"C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\Latest Date\農芸\PKTR022_農芸_202607.csv"

In [56]:
df.to_csv(file_latest, index=False, sep='\t', encoding="utf-8-sig")

## Count rows in each org code

In [57]:
df_fuji = pd.read_csv(
    file_latest,
    delimiter="\t",
    encoding="utf-8-sig",
    dtype={"組織コード": str},
    keep_default_na=False
)

rows_by_org = (
    df_fuji.groupby("組織コード")
    .size()
    .reset_index(name="row_count")
    .sort_values("row_count")
    .reset_index(drop=True)
)

display(rows_by_org)

,組織コード,row_count
0,006000000,77
1,006090100,77
2,006090200,77
3,230F10100,77
4,230V10100,77


## For Agri Only

### Print files in a group of row count

In [14]:
df_latest = pd.read_csv(
    file_latest,
    delimiter="\t",
    encoding="utf-8-sig",
    dtype={"組織コード": str},
    keep_default_na=False
)

org_code_df = (
    df_latest.groupby("組織コード")
    .size()
    .reset_index(name="rows")
)

grouped_codes = (
    org_code_df.groupby("rows")["組織コード"]
    .apply(lambda s: sorted(set(s)))
    .reset_index(name="org_codes")
    .sort_values("rows")
    .reset_index(drop=True)
)

for _, r in grouped_codes.iterrows():
    print(f"Rows = {int(r['rows'])} ({len(r['org_codes'])} unique 組織コード)")
    for code in r["org_codes"]:
        print(f"  - {code}")
    print()

Rows = 70 (31 unique 組織コード)
  - 003030100
  - 003095320
  - 003095420
  - 040A30101
  - 040A30201
  - 100A30101
  - 210A30200
  - 220C30101
  - 230A30101
  - 230B10100
  - 230B30100
  - 230B30200
  - 230B30300
  - 230B30601
  - 230B50100
  - 230C30200
  - 230C30310
  - 230C30330
  - 230C30400
  - 230C30500
  - 230C30600
  - 230H30200
  - 230J30100
  - 230K10300
  - 230K30101
  - 230P10100
  - 230P30101
  - 230Q30101
  - 230S30101
  - 400A10100
  - 400A30100

Rows = 71 (29 unique 組織コード)
  - 003060100
  - 003090100
  - 003090200
  - 003091300
  - 003091500
  - 003092100
  - 003095310
  - 003095410
  - 040A10100
  - 040A10200
  - 100A10200
  - 210A10300
  - 220C10100
  - 230A10200
  - 230B10200
  - 230B60100
  - 230H10100
  - 230H10200
  - 230H10500
  - 230H10700
  - 230H61100
  - 230H61210
  - 230H61600
  - 230H62000
  - 230K10100
  - 230P10300
  - 230Q10200
  - 230S10200
  - 400A10200

Rows = 87 (18 unique 組織コード)
  - 003000000
  - 003010100
  - 040A10300
  - 040A10400
  - 100A10100
  - 

### Check if essential 組織コード is enough or lack 

In [15]:
# Check row count consistency by (対象年月度, 組織コード)
df_check = pd.read_csv(
    file_latest,
    delimiter="\t",
    encoding="utf-8-sig",
    dtype={"組織コード": str}
)

codes_70 = [
'003030100','003095320','003095420','040A30101','040A30201','100A30101',
'210A30200','220C30101','230A30101','230B10100','230B30100','230B30200',
'230B30300','230B30601','230B50100','230C30200','230C30310','230C30330',
'230C30400','230C30500','230C30600','230H30200','230J30100','230K10300',
'230K30101','230P10100','230P30101','230Q30101','230S30101','400A10100',
'400A30100'
]

codes_71 = [
'003060100','003090100','003090200','003091300','003091500','003092100',
'003095310','003095410','040A10100','040A10200','100A10200','210A10300',
'220C10100','230A10200','230B10200','230B60100','230H10100','230H10200',
'230H10500','230H10700','230H61100','230H61210','230H61600','230H62000',
'230K10100','230P10300','230Q10200','230S10200','400A10200'
]

codes_87 = [
'003000000','003010100','040A10300','040A10400','100A10100','100A10300',
'220C10200','220C10300','230A10300','230A10400','230B10300','230H10400',
'230H10600','230K10200','230P10400','230Q10300','230S10300','400A10300'
]

expected_rows = {c: 70 for c in codes_70}
expected_rows.update({c: 71 for c in codes_71})
expected_rows.update({c: 87 for c in codes_87})

actual = (
    df_check.groupby(["対象年月度", "組織コード"])
    .size()
    .reset_index(name="actual_rows")
)
actual["expected_rows"] = actual["組織コード"].map(expected_rows)

mismatch = actual[
    actual["expected_rows"].isna() | (actual["actual_rows"] != actual["expected_rows"])
].sort_values(["組織コード", "対象年月度"]).reset_index(drop=True)

print(f"Checked records: {len(actual)} (年月度 × 組織コード)")
print(f"Mismatches found: {len(mismatch)}")

if mismatch.empty:
    print("All 組織コード row counts match expected (67/71/84).")
else:
    print("Mismatch details:")
    display(mismatch)

    unknown_codes = sorted(mismatch.loc[mismatch["expected_rows"].isna(), "組織コード"].unique().tolist())
    if unknown_codes:
        print(f"Codes not in expected lists ({len(unknown_codes)}): {unknown_codes}")

Checked records: 78 (年月度 × 組織コード)
Mismatches found: 0
All 組織コード row counts match expected (67/71/84).


### Show mismatch details at unique 組織コード level only (ignore 対象年月度)

In [16]:
if mismatch.empty:
    print("No mismatches.")
else:
    mismatch_unique_codes = sorted(mismatch["組織コード"].dropna().astype(str).unique().tolist())
    print(f"Unique mismatched 組織コード: {len(mismatch_unique_codes)}")
    display(pd.DataFrame({"組織コード": mismatch_unique_codes}))

No mismatches.


### Remove unecessary 組織コード

In [8]:
# Remove specific 組織コード rows from the target file, then overwrite the same file
remove_codes = {"003010100", "100A10100", "230A10100", "230H10400"}

df_src = pd.read_csv(
    check_file,  # already set to: ...\アグリ_202204_202511.csv
    delimiter="\t",
    encoding="utf-8-sig",
    dtype={"組織コード": str}
)

before_rows = len(df_src)
df_clean = df_src[~df_src["組織コード"].isin(remove_codes)].copy()
removed_rows = before_rows - len(df_clean)

df_clean.to_csv(check_file, index=False, sep="\t", encoding="utf-8-sig")

print(f"Removed rows: {removed_rows}")
print(f"Remaining rows: {len(df_clean)}")
print(f"Updated file: {check_file}")

Removed rows: 252
Remaining rows: 4735
Updated file: C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\Latest Date\アグリ\PKTR022_アグリ_４.csv


# Add Columns

## Concat files

In [58]:
import pandas as pd

# Load the CSV file into df_flo with proper delimiter
df = pd.read_csv(file_latest, 
                 delimiter='\t', encoding="utf-8-sig", dtype={'組織コード': str})

#Filter 組織コード

target_org_codes = ['006000000', '006090100', '006090200', '230F10100', '230V10100']


df = df[df['組織コード'].isin(target_org_codes)]

# Add 組織名 column using mapping

mapping = {'006000000': 'イノチオ農芸㈱',
 '006090100': 'イノチオ農芸本社',
 '006090200': '本社収支',
 '230F10100': '種苗課',
 '230V10100': '農芸トマト課'}



df['組織名'] = df['組織コード'].map(mapping).fillna('不明な組織')

 # Add 分類 column
def get_category(row):
    #For other 会社　
     if row['明細No.'] == 1:
         return "受注"
     elif 3 <= row['明細No.'] <= 7:
         return "売上"
     elif 17 <= row['明細No.'] <= 70 and row['明細No.'] != 36:
           return "経費"
     elif 74 <= row['明細No.'] <= 76:
         return "時間"
     else:
         return None

    #--------------------------

    # For Agri 
    # n = expected_rows.get(row['組織コード'])
    # d = row['明細No.']

    # if n == 70:
    #     if 9 <= row['明細No.'] <= 61 and row['明細No.'] != 15:
    #      return "経費"
    #     elif 65 <= row['明細No.'] <= 68:
    #         return "時間"
    #     else:
    #         return None

    # elif n == 71:
    #     if 2 <= d <= 4:
    #         return "受注"
    #     elif 6 <= d <= 11:
    #         return "売上"
    #     elif 19 <= d <= 62:
    #         return "経費"
    #     elif 66 <= d <= 69:
    #         return "時間"
    #     else:
    #         return None

    # elif n == 87:
    #     if 2 <= row['明細No.'] <= 4:
    #      return "受注"
    #     elif 6 <= row['明細No.'] <= 11:
    #      return "売上"
    #     elif 26 <= row['明細No.'] <= 78 and row['明細No.'] != 32:
    #      return "経費"
    #     elif 82 <= row['明細No.'] <= 85:
    #      return "時間"
    #     else:
    #      return None

    # return None
    

df['分類'] = df.apply(get_category, axis=1)

# Add 採算科目名_経費合計 based on calculated 分類
df['採算科目名_経費合計'] = df['採算科目名'].where(df['分類'] == '経費')

In [59]:
df.head(10)

,対象年月度,組織コード,明細No.,採算科目名,M/P,予定,実績,構成比(%),予定比(%),M/P比(%),組織名,分類,採算科目名_経費合計
0,202607,006000000,1,総受注高,15500000.0,14782100.0,14929391.0,NaN,100.9,96.3,イノチオ農芸㈱,受注,NaN
1,202607,006000000,2,総売上高,15500000.0,14782100.0,14929391.0,NaN,100.9,96.3,イノチオ農芸㈱,None,NaN
2,202607,006000000,3,売上高,12020000.0,10000000.0,10016182.0,67.0,100.1,83.3,イノチオ農芸㈱,売上,NaN
3,202607,006000000,4,グループ売上高,3480000.0,4782100.0,4913209.0,32.9,102.7,141.1,イノチオ農芸㈱,売上,NaN
4,202607,006000000,5,社内売上高,0.0,0.0,0.0,0.0,0.0,0.0,イノチオ農芸㈱,売上,NaN
5,202607,006000000,7,販売奨励金,0.0,0.0,0.0,0.0,0.0,0.0,イノチオ農芸㈱,売上,NaN
6,202607,006000000,8,売上原価,150000.0,150000.0,119800.0,0.8,79.8,79.8,イノチオ農芸㈱,None,NaN
7,202607,006000000,9,仕入高,0.0,0.0,0.0,0.0,0.0,0.0,イノチオ農芸㈱,None,NaN
8,202607,006000000,10,グループ仕入高,0.0,0.0,0.0,0.0,0.0,0.0,イノチオ農芸㈱,None,NaN
9,202607,006000000,11,社内仕入高,0.0,0.0,0.0,0.0,0.0,0.0,イノチオ農芸㈱,None,NaN


## Combine with the full monthly files

In [60]:
import pandas as pd

file_name = r'C:\Users\2372\OneDrive - イノチオホールディングス株式会社\採算表_RPA\実績\Added Columns\PKTR022_農芸.csv'

original_df = pd.read_csv(file_name, delimiter='\t', encoding="utf-8-sig")

combined_df = pd.concat([df, original_df], ignore_index=True)
combined_df.to_csv(file_name, index=False, sep='\t', encoding="utf-8-sig")

# Create 分類 table

In [ ]:
# import pandas as pd

# # Create the table as a DataFrame
# table_data = {
#     "分類": ["受注", "売上", "収益", "経費", "時間"],
#     "分類順": [1, 2, 3, 4, 5]
# }
# table_df = pd.DataFrame(table_data)

# # Save to CSV
# table_df.to_csv("Added Columns/分類Table.csv", index=False, sep='\t', encoding="utf-8-sig")